# Module 1 — Document Ingestion
**Day 1 | ~60 minutes**

**Goal:** Load real DKV Belgium documents, understand their structure and language mix,  
and prepare them for chunking.

---

## What we're working with

DKV Belgium insurance documents are:
- **Multilingual** — the same policy exists in French (FR), Dutch (NL), and sometimes English (EN)
- **Structured** — articles, sections, sub-sections with clear numbering
- **Dense** — lots of legal and medical terminology (INAMI, RIZIV, NIHDI)
- **PDF-first** — the primary format is PDF, not plain text

Each of these properties affects how we chunk, embed, and retrieve.

---
## Exercise 0 — Implement the document loader

Before running the cells below, open [`src/document_loader.py`](../src/document_loader.py) and implement two functions:

### `_detect_language(sample: str) -> str`
For each language's stopword set, count how many words appear in `sample_lower`.
Wrap each word with spaces (`" word "`) to avoid partial matches.
Return the language key (`"fr"`, `"nl"`, `"en"`) with the highest count,
or `"unknown"` if all scores are zero.

### `load_documents(directory, extensions) -> list[dict]`
The `loaders` dict and skeleton are already there. You need to:
1. For each extension, glob recursively: `directory.glob(f"**/*{ext}")`
2. Skip files starting with `"."` or named `"README.txt"`
3. Call the matching loader; skip docs with empty content
4. Add `"subfolder"` metadata (`path.parent.name` if nested, else `""`)
5. Use the subfolder prefix (`EN_`/`FR_`/`NL_`) to override heuristic language detection
6. Print each file as `[LANG] subfolder/filename (N chars)`
7. Wrap each loader call in `try/except` and print a warning on failure

Once both functions are implemented, run the sanity-check cell below, then continue.

In [ ]:
# Quick sanity check — test _detect_language before loading the full corpus
import sys
sys.path.insert(0, "..")
from src.document_loader import _detect_language

tests = [
    ("L'assureur prend en charge les frais d'hospitalisation selon les conditions.", "fr"),
    ("De verzekeraar dekt de ziekenhuiskosten voor de verzekerde personen.", "nl"),
    ("The insurer covers hospitalisation costs for the insured persons.", "en"),
]

all_ok = True
for text, expected in tests:
    result = _detect_language(text[:500])
    status = "\u2713" if result == expected else "\u2717"
    if result != expected:
        all_ok = False
    print(f"{status}  expected={expected}  got={result}  | {text[:60]}")

if all_ok:
    print("\n\u2713 Language detection works \u2014 ready to load documents.")
else:
    print("\n\u2717 Fix _detect_language before continuing.")

In [ ]:

from pathlib import Path
import json

from src.document_loader import load_documents

# Directory with DKV documents
DOCS_DIR = "../data/sample_dkv"

print(f"Loading documents from: {Path(DOCS_DIR).resolve()}")
documents = load_documents(DOCS_DIR)
print(f"\nLoaded {len(documents)} documents")

---
## Step 1 — Explore the corpus

In [ ]:
import pandas as pd

# Overview of loaded documents
doc_df = pd.DataFrame([
    {
        "title": d["title"],
        "language": d["language"].upper(),
        "format": d["format"],
        "chars": len(d["content"]),
        "words": len(d["content"].split()),
        "paragraphs": d["content"].count("\n\n"),
    }
    for d in documents
])

print(doc_df.to_string(index=False))
print(f"\nTotal: {doc_df['words'].sum():,} words across {len(documents)} documents")
print(f"Language breakdown: {doc_df['language'].value_counts().to_dict()}")

In [ ]:
# Inspect a document — read the first 1000 characters
doc = documents[0]
print(f"Document: {doc['title']}")
print(f"Language: {doc['language'].upper()} | Format: {doc['format']}")
print("\n" + "─" * 60)
print(doc["content"][:1500])
print("─" * 60)
print(f"\n... ({len(doc['content'])} total characters)")

In [ ]:
# Inspect document structure — language-agnostic approach
# Instead of matching specific keywords, find lines that are structurally different:
# short lines relative to the document average are usually headers or article titles.

for doc in documents:
    lines = [l.strip() for l in doc["content"].split("\n") if l.strip()]
    if not lines:
        continue

    avg_len = sum(len(l) for l in lines) / len(lines)

    # Structural candidates: 2–12 words AND shorter than 60% of average line length
    candidates = [
        l for l in lines
        if 2 <= len(l.split()) <= 12
        and len(l) < avg_len * 0.65
        and l[-1] not in {",", ";", "-"}  # not mid-sentence
    ]

    print(f"\n{doc['title']} [{doc['language'].upper()}] | {doc['format']}")
    print(f"  {len(lines)} lines, avg {avg_len:.0f} chars/line → {len(candidates)} structural candidates")
    for c in candidates[:8]:
        print(f"    · {c[:80]}")

print("\n" + "─" * 60)
print("Tip: PDF docs often have fewer clean headers because pdfplumber")
print("     preserves the visual layout — lots of spaces, no real newlines.")
print("     TXT docs have explicit paragraph breaks (\\n\\n) that chunk naturally.")

In [ ]:
# Deep-read one document: print every non-empty line with its word count
# Change the index to inspect a different document
TARGET = 3  # try 0 (PDF) vs 3 (TXT) to see the difference

doc = documents[TARGET]
print(f"Full line-by-line view: {doc['title']} [{doc['language'].upper()}] ({doc['format']})")
print("─" * 70)

lines = [l.strip() for l in doc["content"].split("\n") if l.strip()]
for i, line in enumerate(lines[:60]):
    wc = len(line.split())
    marker = "◀ short" if wc <= 8 else ""
    print(f"{i:>3}  [{wc:>3}w]  {line[:80]}  {marker}")

if len(lines) > 60:
    print(f"\n... ({len(lines) - 60} more lines not shown)")

---
## Step 2 — Verify multilingual coverage

In [ ]:
# Step 2 — Verify multilingual coverage
#
# WHY: before chunking and indexing, confirm that:
#   1. Language detection assigned the right language to each document
#   2. The same policy content is present in FR, NL, and EN
#   3. PDF extraction didn't produce garbled text
#
# If a document loads with the wrong language or corrupted text, fix it NOW
# rather than discovering it through bad retrieval scores.

# Show the opening paragraph of each language version of the hospitalisation policy
hosp_docs = [
    d for d in documents
    if "hospitalisation" in d["title"].lower() or "ziekenhuisopname" in d["title"].lower()
]

for doc in sorted(hosp_docs, key=lambda d: d["language"]):
    # First non-empty paragraph (skip title lines)
    paragraphs = [p.strip() for p in doc["content"].split("\n\n") if len(p.strip().split()) > 10]
    excerpt = paragraphs[0][:400] if paragraphs else doc["content"][:400]

    print(f"[{doc['language'].upper()}] {doc['title']}")
    print(excerpt)
    print()

print("─" * 60)
print("Check: do all three versions describe the same policy?")
print("Check: is each language label correct?")

---
## Step 3 — Save the corpus for use in later notebooks

> **Workshop convenience only.** We serialise the loaded documents to a JSON file so that
> later notebooks can reload them without re-running PDF extraction. In production you would
> never do this: raw documents stay in object storage (Azure Blob / S3) as the source of truth,
> and extracted text flows directly into the vector store or a document database.

In [ ]:
# Workshop convenience: serialise to JSON so later notebooks can reload without re-running extraction.
# In production, extracted text goes directly into the vector store — raw PDFs stay in object storage.
corpus_path = Path("../data/raw/corpus.json")
corpus_path.parent.mkdir(parents=True, exist_ok=True)

with open(corpus_path, "w", encoding="utf-8") as f:
    json.dump(documents, f, ensure_ascii=False, indent=2)

print(f"Saved {len(documents)} documents to {corpus_path}")
print("Ready for Module 2: chunking and embedding")

---
## Discussion: What makes these documents challenging?

Before moving to chunking, discuss:

1. **Mixed languages in one PDF** — a document may have FR and NL sections.  
   How should we handle that at indexing time?

2. **Tables** — insurance documents often use tables for benefit schedules.  
   pdfplumber can extract tables; pypdf cannot. Did you notice any table-related issues above?

3. **Section structure** — articles and sub-sections are semantic boundaries.  
   Should our chunking respect them? (Foreshadowing Module 2...)

4. **Legal terminology** — `INAMI`, `RIZIV`, `NIHDI` are the same organisation in three languages.  
   Will our embedding model know that?

---

## Next: Module 2 — Chunking & Embedding

We have documents loaded and understood. Now we split them into retrievable pieces  
and convert them to vectors. Open `02_chunking_and_embedding.ipynb` →